# 🔍 Web Scraping: Buscador 23F - RTVE Desclasificados


---
> ⚠️ **Aviso legal**: Este scraping es para uso personal/investigación. Respeta el `robots.txt` y los Términos de Uso de RTVE. No sobrecargues sus servidores.

In [ ]:
!pip install -q requests beautifulsoup4 lxml pandas tqdm

In [ ]:
import json, re, time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

BASE = "https://23fbuscador.rtve.es"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; 23F-research-scraper/1.0)",
    "Accept-Language": "es-ES,es;q=0.9",
}
session = requests.Session()
session.headers.update(HEADERS)

OUTPUT_DIR = Path("data_23f")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Guardaré los datos en:", OUTPUT_DIR.resolve())

Guardaré los datos en: /content/data_23f


In [ ]:
def fetch_listing_page(page=1, page_size=200):
    url = f"{BASE}/?page_size={page_size}&page={page}"
    r = session.get(url, timeout=30); r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")

def parse_listing(soup):
    rows = []
    for tr in soup.select("table tbody tr"):
        tds = tr.find_all("td")
        if len(tds) < 5: continue
        link = tds[0].find("a")
        if not link: continue
        href = link.get("href", "")
        m = re.search(r"/document/ocr/(\d+)", href)
        rows.append({
            "id": int(m.group(1)) if m else None,
            "titulo": link.get_text(strip=True),
            "detalle_url": urljoin(BASE, href.split("?")[0]),
            "paginas": tds[1].get_text(strip=True),
            "kb": tds[2].get_text(strip=True),
            "resumen_breve": tds[3].get_text(" ", strip=True),
            "tags_breve": [c.get_text(strip=True) for c in tds[4].select(".tag-chip")],
        })
    return rows

def total_pages(soup):
    txt = soup.select_one(".nav-position")
    if not txt: return 1
    m = re.search(r"de\s+(\d+)", txt.get_text())
    return int(m.group(1)) if m else 1

def collect_all_documents(page_size=200):
    soup = fetch_listing_page(1, page_size)
    docs = parse_listing(soup)
    pages = total_pages(soup)
    for p in range(2, pages + 1):
        soup = fetch_listing_page(p, page_size)
        docs.extend(parse_listing(soup))
        time.sleep(0.5)
    return docs

documents = collect_all_documents()
print(f"Documentos encontrados: {len(documents)}")
documents[:2]

Documentos encontrados: 167


[{'id': 1860,
  'titulo': 'Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).',
  'detalle_url': 'https://23fbuscador.rtve.es/document/ocr/1860',
  'paginas': '3',
  'kb': '-',
  'resumen_breve': 'El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con declaraciones parciales de altos mandos militares y certificaciones oficiales, aunque plagado de controversias por la interpretación y selección de testimonios, especialmente en re...',
  'tags_breve': ['C/SG/2820/20-02-82', 'DTOR.', 'Vista oral 2/81']},
 {'id': 1859,
  'titulo': 'Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).',
  'detalle_url': 'https://23fbuscador.rtve.es/document/ocr/1859',
  'paginas': '4',
  'kb': '-',
  'resumen_breve': 'Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febrero de 1982 por el Consejo Supremo de Justicia Militar,

In [ ]:
def parse_detail(html, url):
    soup = BeautifulSoup(html, "lxml")
    titulo_el = soup.select_one("header.page-header h2")
    titulo = titulo_el.get_text(strip=True) if titulo_el else None

    meta = {}
    for div in soup.select(".detail-grid div"):
        strong = div.find("strong")
        if not strong: continue
        key = strong.get_text(strip=True).rstrip(":").strip().lower()
        strong.extract()
        val = div.get_text(" ", strip=True).lstrip(":").strip()
        meta[key] = val

    def section_with_heading(heading):
        for sec in soup.select(".detail-section"):
            h = sec.find("h3")
            if h and h.get_text(strip=True).lower() == heading.lower():
                return sec
        return None

    resumen = None
    sec = section_with_heading("Resumen")
    if sec:
        p = sec.find("p")
        resumen = p.get_text("\n", strip=True) if p else None

    personas, lugares, palabras_clave = [], [], []
    sec = section_with_heading("Palabras clave")
    if sec:
        for card in sec.select(".tag-group-card"):
            cls = card.get("class", [])
            chips = [c.get_text(strip=True) for c in card.select(".tag-chip")]
            if "tag-group-people" in cls: personas = chips
            elif "tag-group-places" in cls: lugares = chips
            elif "tag-group-keywords" in cls: palabras_clave = chips

    pre = soup.select_one("pre.text-box-large, pre.text-box")
    texto_completo = pre.get_text("\n", strip=False) if pre else None

    m = re.search(r"/document/ocr/(\d+)", url)
    doc_id = int(m.group(1)) if m else None

    return {
        "id": doc_id, "titulo": titulo, "detalle_url": url,
        "original_url": f"{BASE}/document/ocr/{doc_id}/original" if doc_id else None,
        "tipo": meta.get("tipo"), "estado": meta.get("estado"),
        "paginas": meta.get("páginas") or meta.get("paginas"),
        "kb": meta.get("kb"),
        "modelo_ocr": meta.get("modelo"), "proveedor_ocr": meta.get("proveedor"),
        "resumen": resumen,
        "personas": personas, "lugares": lugares, "palabras_clave": palabras_clave,
        "texto_completo": texto_completo,
    }

def fetch_detail(url, retries=3, backoff=1.5):
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, timeout=30); r.raise_for_status()
            return parse_detail(r.text, url)
        except requests.RequestException:
            if attempt == retries: raise
            time.sleep(backoff ** attempt)

# Prueba con el primer documento
sample = fetch_detail(documents[0]["detalle_url"])
{k: (v[:120] + "..." if isinstance(v, str) and len(v) > 120 else v) for k, v in sample.items()}

{'id': 1860,
 'titulo': 'Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).',
 'detalle_url': 'https://23fbuscador.rtve.es/document/ocr/1860',
 'original_url': 'https://23fbuscador.rtve.es/document/ocr/1860/original',
 'tipo': 'ocr',
 'estado': 'ok',
 'paginas': '3',
 'kb': '-',
 'modelo_ocr': 'mistral-ocr-latest',
 'proveedor_ocr': 'mistral',
 'resumen': 'El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, con ...',
 'personas': ['No consta',
  'Luis Arana Lorite:Teniente Coronel',
  'Manuel Miler Hidalgo:Teniente Coronel',
  'TEJERO:Teniente Coronel',
  'SR. CARRES',
  'Capitán GOMEZ IGLESIAS',
  'Sr. LAVILLA:Presidente del Congreso',
  'DIEGO IBAÑEZ INGLES:Coronel',
  'José María Fernández del Rio Fernández:Gobernador Civil de Valencia',
  'MAS OLIVER:Teniente Coronel'],
 'lugares': ['Valencia',
  'Congreso (Presidente del Congreso)',
  'Gobernador Civil de Valencia',
  'SALA',
  'Ministeri

In [ ]:
details, errores = [], []
for d in tqdm(documents, desc="Descargando detalles"):
    try:
        details.append(fetch_detail(d["detalle_url"]))
    except Exception as exc:
        errores.append({"id": d["id"], "url": d["detalle_url"], "error": str(exc)})
    time.sleep(0.4)

print(f"OK: {len(details)} | errores: {len(errores)}")
errores[:5]

Descargando detalles:   0%|          | 0/167 [00:00<?, ?it/s]

OK: 167 | errores: 0


[]

In [ ]:
# Guardar
json_path = OUTPUT_DIR / "documentos_23f.json"
json_path.write_text(json.dumps(details, ensure_ascii=False, indent=2), encoding="utf-8")

df = pd.DataFrame(details)
for col in ["personas", "lugares", "palabras_clave"]:
    df[col] = df[col].apply(lambda xs: " | ".join(xs) if isinstance(xs, list) else "")

csv_path = OUTPUT_DIR / "documentos_23f.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

txt_dir = OUTPUT_DIR / "transcripciones"; txt_dir.mkdir(exist_ok=True)
for d in details:
    if d.get("texto_completo"):
        (txt_dir / f"{d['id']}.txt").write_text(d["texto_completo"], encoding="utf-8")

# Empaquetar todo en un zip y descargarlo desde Colab
import shutil
zip_path = shutil.make_archive("data_23f", "zip", OUTPUT_DIR)
print("ZIP creado:", zip_path)

from google.colab import files
files.download(zip_path)

ZIP creado: /content/data_23f.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>